## nb_backfill_buffer

**One-time backfill** — run this once after uploading the 2022 parquet and before enabling
the `pl_hourly_predict` pipeline trigger.

The inference notebook needs 6 hours of history for the sequence input, and the lag features
in the graph (`ARR_DELAY_LAGS = [1, 3, 6, 24]`) need up to 24 hours of past data.
Without backfilling, the first ~24 hours of predictions will have degraded accuracy
because the T-24h lag feature will be zero.

This notebook calls the ingestion logic for the past `BACKFILL_HOURS` hours
so the rolling buffer is fully populated from the first prediction onward.

**Typical usage:** set `BACKFILL_HOURS = 168` (7 days) and run once.
Runtime: ~168 iterations × ~3s each ≈ 8 minutes.

In [ ]:
BACKFILL_HOURS = 168  # How many hours back to fill (168 = 7 days)
SOURCE_YEAR    = 2022

In [ ]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

LAKEHOUSE_ROOT = Path("/lakehouse/default/Files")
SOURCE_PARQUET = LAKEHOUSE_ROOT / "raw" / "historical_2022" / "Combined_Flights_2022.parquet"
BUFFER_ROOT    = LAKEHOUSE_ROOT / "live_feed" / "rolling_buffer"
CLASS_B_ROOT   = LAKEHOUSE_ROOT / "live_feed" / "class_b"

if not SOURCE_PARQUET.exists():
    raise FileNotFoundError(f"Upload the 2022 parquet first: {SOURCE_PARQUET}")

parquet_schema = pq.read_schema(SOURCE_PARQUET)
available_cols = set(parquet_schema.names)

CLASS_A_COLS = [
    "FlightDate", "Airline", "Origin", "Dest",
    "CRSDepTime", "CRSArrTime", "CRSElapsedTime", "Distance",
    "Month", "DayOfWeek", "DayofMonth", "Cancelled", "Diverted",
]
CLASS_B_COLS = CLASS_A_COLS + [
    "DepTime", "ArrTime", "DepDelay", "ArrDelay",
    "DepDel15", "ArrDel15", "WheelsOff", "WheelsOn",
    "TaxiOut", "TaxiIn", "AirTime", "ActualElapsedTime",
    "CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay",
]
b_cols = [c for c in CLASS_B_COLS if c in available_cols]

now_utc = pd.Timestamp.utcnow().floor("h")
print(f"Backfilling {BACKFILL_HOURS} hours ending at {now_utc}")
print(f"This will populate {BACKFILL_HOURS} partitions in rolling_buffer/ and class_b/")

In [ ]:
errors = []

for offset in range(BACKFILL_HOURS, 0, -1):
    target_ts  = now_utc - pd.Timedelta(hours=offset)
    equiv_dt   = target_ts.replace(year=SOURCE_YEAR)

    hour_start = equiv_dt.hour * 100
    hour_end   = (equiv_dt.hour + 1) * 100

    try:
        table = pq.read_table(
            str(SOURCE_PARQUET),
            columns=b_cols,
            filters=[
                ("Month",      "=",  equiv_dt.month),
                ("DayofMonth", "=",  equiv_dt.day),
                ("CRSDepTime", ">=", hour_start),
                ("CRSDepTime", "<",  hour_end),
            ],
        )

        for root in (BUFFER_ROOT, CLASS_B_ROOT):
            out_dir = root / f"{target_ts.year}/{target_ts.month:02d}/{target_ts.day:02d}/{target_ts.hour:02d}"
            out_dir.mkdir(parents=True, exist_ok=True)
            pq.write_table(table, str(out_dir / "flights.parquet"), compression="snappy")

        if offset % 24 == 0 or offset <= 5:
            print(f"  [{BACKFILL_HOURS - offset + 1}/{BACKFILL_HOURS}] {target_ts} → {len(table):,} rows")

    except Exception as e:
        errors.append((target_ts, str(e)))
        print(f"  ERROR at {target_ts}: {e}")

print(f"\nBackfill complete. {BACKFILL_HOURS - len(errors)}/{BACKFILL_HOURS} hours written successfully.")
if errors:
    print(f"Failed hours: {[str(ts) for ts, _ in errors]}")
    print("Re-run with BACKFILL_HOURS set to cover only the failed range.")
else:
    print("Rolling buffer is fully populated. You can now enable pl_hourly_predict.")